## Phase 1: Extract

In [1]:
import pandas as pd
import sqlite3

# โหลดข้อมูลดิบ
df_raw = pd.read_csv('raw_ecommerce_data.csv')

#ตรวจสอบโครงสร้างและค่า Null
df_raw.info()
df_raw.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Order_ID       185 non-null    object
 1   Customer_Name  183 non-null    object
 2   Email          184 non-null    object
 3   Product        185 non-null    object
 4   Category       184 non-null    object
 5   Order_Date     185 non-null    object
 6   Quantity       185 non-null    int64 
 7   Unit_Price     185 non-null    object
 8   Amount         142 non-null    object
dtypes: int64(1), object(8)
memory usage: 13.1+ KB


,Order_ID,Customer_Name,Email,Product,Category,Order_Date,Quantity,Unit_Price,Amount
0,ORD-0036,Emma Brown,emma.brown@email.com,Monitor 24 inch,Electronics,24/04/2026,1,4131.00,"4,131.00"
1,ORD-0076,linda park,linda.park@email.com,Mechanical Keyboard,Electronics,"Apr 03, 2026",5,1669.50,8347.50
2,ORD-0101,john doe,john@email.com,Wireless Mouse,Electronics,11/04/2026,3,405.00,NaN
3,ORD-0059,Jane Smith,jane@email.com,Wireless Mouse,ELECTRONICS,13/03/2026,2,405.00,฿810.00
4,ORD-0038,PETER KIM,peter.kim@email.com,Gel Pen Set,Stationery,2026-02-28,5,85.50,427.50


## Phase 2A:  Transform Built Dimention Table



In [3]:
# 1. ดึงเฉพาะคอลัมน์ที่เกี่ยวกับลูกค้า และลบแถวที่ข้อมูลลูกค้าซ้ำกันสมบูรณ์แบบ
dim_customer = df_raw[['Customer_Name', 'Email']].drop_duplicates()

# 2. จัดการ Missing Value (แทนที่ NaN ด้วยค่าเริ่มต้น)
dim_customer['Customer_Name'] = dim_customer['Customer_Name'].fillna('Unknown Customer')
dim_customer['Email'] = dim_customer['Email'].fillna('No Email')

# 3. ลบซ้ำอีกรอบ (กรณีที่เคยมี NaN หลายอัน แล้วถูกเปลี่ยนเป็น 'Unknown Customer' เหมือนกัน)
dim_customer = dim_customer.drop_duplicates()

# 4. สร้าง Surrogate Key (customer_id)
dim_customer = dim_customer.reset_index(drop=True)
dim_customer['customer_id'] = dim_customer.index + 1

# 5. จัดเรียงคอลัมน์ให้ PK อยู่หน้าสุด
dim_customer = dim_customer[['customer_id', 'Customer_Name', 'Email']]

print(dim_customer.head(10))

   customer_id Customer_Name                 Email
0            1   Emma Brown   emma.brown@email.com
1            2    linda park  linda.park@email.com
2            3      john doe       john@email.com 
3            4    Jane Smith        jane@email.com
4            5     PETER KIM   peter.kim@email.com
5            6   Jane Smith         jane@email.com
6            7   Michael Tan   michael.t@email.com
7            8      Nok Kwan    nok.kwan@email.com
8            9      Krit Som    krit.som@email.com
9           10    Emma Brown  emma.brown@email.com


## Phase 2B: Transform built Fact Table

In [4]:
# นำ customer_id กลับไปใส่ใน fact table ผ่านการ Left Join (ตามสไลด์)
fact_sales = pd.merge(
    df_raw,
    dim_customer,
    on=['Customer_Name', 'Email'],
    how='left'
)

# ลบคอลัมน์ Text ทิ้ง เหลือไว้เพียง Foreign Key
fact_sales = fact_sales.drop(columns=['Customer_Name', 'Email'])

## Phase 3A: Load by setting SQLite Warehouse

In [5]:
# 1. สร้าง Connection ไปที่ไฟล์ .db
conn = sqlite3.connect('warehouse.db')
cursor = conn.cursor()

# 2. สร้างตาราง Dimension พร้อมกำหนด Primary Key
cursor.execute('''
    CREATE TABLE IF NOT EXISTS dim_customer (
        customer_id INTEGER PRIMARY KEY,
        Customer_Name TEXT,
        Email TEXT
    )
''')
conn.commit()

## Phase 3B: Load force to use Star Schema Relational

In [9]:
conn = sqlite3.connect('warehouse.db')
cursor = conn.cursor()

# เปิดใช้งานการตรวจสอบ Foreign Key ใน SQLite
cursor.execute('PRAGMA foreign_keys = ON;')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS fact_sales (
        transaction_id INTEGER PRIMARY KEY,
        customer_id INTEGER,
        product_id INTEGER,
        amount REAL,
        FOREIGN KEY (customer_id) REFERENCES dim_customer(customer_id),
        FOREIGN KEY (product_id) REFERENCES dim_product(product_id)
    )
''')
conn.commit()

## Phase 3C: Load push data on Warehouse

In [10]:
# โหลดข้อมูล Dimension
dim_customer.to_sql('dim_customer', con=conn,
                    if_exists='replace', index=False)

# โหลดข้อมูล Fact
fact_sales.to_sql('fact_sales', con=conn,
                  if_exists='replace', index=False)

print('ETL Pipeline ran successfully!')

ETL Pipeline ran successfully!


## Verification: Testing Query from Warehouse

In [13]:
# 1. ดึงข้อมูลด้วย SQL
query = '''
SELECT
    c.Customer_Name,
    SUM(f.amount) as Total_Spend
FROM fact_sales f
JOIN dim_customer c ON f.customer_id = c.customer_id
GROUP BY c.Customer_Name
ORDER BY Total_Spend DESC
LIMIT 3;
'''

df_result = pd.read_sql_query(query, conn)

# 2. จัดรูปแบบตัวเลขให้มี Comma และ ทศนิยม 2 ตำแหน่ง
df_result['Total_Spend'] = df_result['Total_Spend'].map('{:,.2f}'.format)

# 3. แสดงผลลัพธ์
print(df_result.to_string(index=False))

Customer_Name Total_Spend
    Narin Dee   37,597.50
   Alice Wong   25,009.50
     Krit Som   23,976.00
